# Lab 5 — Perturbation and Projection

**ECON 282E · Session 5 · October 22, 2026**

Sessions 3 and 4 put values on a grid and iterated. This lab does neither.

| Part | What you build | Deck |
|---|---|---|
| 1 | steady state, log-linearization, and the matrices $A$, $B$ | 5.A4 |
| 2 | the QZ solve, and the Blanchard--Kahn count | 5.A4 |
| 3 | determinacy in a New Keynesian model: the Taylor principle | 5.A4 |
| 4 | second order by residual matching, and the risk correction | 5.A5 |
| 5 | pruning, and when it matters | 5.A5 |
| 6 | Chebyshev: conditioning, nodes, and what smoothness buys | 5.B2--5.B3 |
| 7 | collocation on the same RBC model | 5.B5--5.B6 |
| 8 | one model, four methods | 5.B6 |

**Prerequisite:** L04. Everything is NumPy and SciPy; nothing needs the internet or a GPU.
The measured numbers quoted in the exercises are the lecture's, from
`tools/figures/s05_numbers.json`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import chebyshev as npcheb
from scipy.linalg import ordqz
from scipy.optimize import root

np.set_printoptions(precision=6, suppress=True)

# Quarterly RBC, the calibration of Session 4 and HW1 Part B Q3
ALPHA, BETA, DELTA = 0.33, 0.99, 0.025
RHO, SIG_EPS = 0.95, 0.007

## Part 1 — The steady state, and the linearized system

Perturbation needs one point: the deterministic steady state. Everything else is
derivatives evaluated there.

In [ ]:
def steady_state(alpha=ALPHA, beta=BETA, delta=DELTA):
    k = ((1/beta - 1 + delta)/alpha)**(1/(alpha - 1))
    y = k**alpha
    c = y - delta*k
    return k, y, c

kss, yss, css = steady_state()
kappa = BETA*ALPHA*kss**(ALPHA - 1)
print(f"k* = {kss:.6f}   y* = {yss:.6f}   c* = {css:.6f}   kappa = {kappa:.6f}")
print("4.B5 reported k* = 28.3484 -- the cheapest cross-check available.")

In [ ]:
def growth_matrices(alpha=ALPHA, beta=BETA, delta=DELTA, rho=RHO):
    """A E_t s' = B s,  s = (khat, z, chat).

    Rows: resource constraint, shock process, Euler equation.
    """
    k, y, c = steady_state(alpha, beta, delta)
    kap = alpha*k**(alpha - 1)*beta
    A = np.array([[k,                0.0,   0.0],
                  [0.0,              1.0,   0.0],
                  [-kap*(alpha - 1), -kap,  1.0]])
    B = np.array([[alpha*y + (1 - delta)*k, y,   -c],
                  [0.0,                     rho,  0.0],
                  [0.0,                     0.0,  1.0]])
    return A, B

A, B = growth_matrices()
print("A =\n", A, "\n\nB =\n", B)

**Exercise 1.** Every entry of `A` and `B` is an elasticity at the steady state. Which entry is
$\kappa$, and why is it only $0.035$? (Hint: at $\delta=0.025$, how much of next period's gross
return is undepreciated capital that does not respond to anything?)

## Part 2 — One QZ decomposition

The whole first-order solve. `ordqz` returns the generalized Schur form with the stable
eigenvalues ordered first; Blanchard--Kahn is then a count.

In [ ]:
def klein(A, B, n_x):
    """Solve A E_t s' = B s with s = (x; y), x predetermined.  Klein (2000)."""
    def stable(a, b):
        return np.abs(b) < np.abs(a)          # |omega| = |b/a| < 1

    S, T, aa, bb, Q, Z = ordqz(A, B, sort=stable, output="real")
    omega = np.abs(bb)/np.where(np.abs(aa) < 1e-14, 1e-14, np.abs(aa))
    n_unstable = int(np.sum(omega > 1 + 1e-9))

    Z11, Z21 = Z[:n_x, :n_x], Z[n_x:, :n_x]
    try:
        # If the BK count is wrong, Z11 is singular: there is no P and F to form.
        Z11i = np.linalg.inv(Z11)
        F = Z21 @ Z11i
        P = Z11 @ np.linalg.solve(S[:n_x, :n_x], T[:n_x, :n_x]) @ Z11i
    except np.linalg.LinAlgError:
        P = F = None
    return P, F, np.sort(omega), n_unstable


P, F, omega, n_unst = klein(A, B, 2)
print("eigenvalue moduli :", np.round(omega, 6))
print("outside unit circle:", n_unst, " jump variables:", 1)
print("\nk' on (khat, z):", np.round(P[0], 6))
print("c  on (khat, z):", np.round(F[0], 6))

**Exercise 2.** The lecture reports $0.962061$ and $0.080097$ for capital, $0.590408$ and
$0.322850$ for consumption. Do you reproduce them?

**Exercise 3.** Re-run with `DELTA = 1.0`, `ALPHA = 0.36`, `BETA = 0.95` — the Brock--Mirman
benchmark. The exact policy is $k' = \alpha\beta e^{z}k^{\alpha}$, which in logs is
$\hat{k}' = \alpha\hat{k} + z$. You should get $P[0] = [0.36, 1.00]$ to machine precision, and
the unstable root should be exactly $1/(\alpha\beta) = 2.923977$. **This is a correctness test,
not an accuracy test** — see 5.A2.

In [ ]:
A1, B1 = growth_matrices(alpha=0.36, beta=0.95, delta=1.0)
P1, F1, om1, nu1 = klein(A1, B1, 2)
print("P =", np.round(P1[0], 12), "   exact: [0.36, 1.0]")
print("F =", np.round(F1[0], 12), "   exact: [0.36, 1.0]")
print("roots:", np.round(om1, 6), "   1/(alpha*beta) =", round(1/(0.36*0.95), 6))

## Part 3 — Determinacy: the Taylor principle as an eigenvalue count

The growth model can only ever satisfy Blanchard--Kahn. To see the count actually bite, take the
three-equation New Keynesian model with two jump variables $(\pi, x)$ — so uniqueness needs
**two** unstable roots.

In [ ]:
def nk_matrices(phi_pi, kappa=0.1275, sigma_is=1.0, rho_u=0.8, beta=BETA):
    A = np.array([[1.0, 0.0,          0.0],
                  [0.0, beta,         0.0],
                  [0.0, 1.0/sigma_is, 1.0]])
    B = np.array([[rho_u, 0.0,             0.0],
                  [0.0,   1.0,            -kappa],
                  [0.0,   phi_pi/sigma_is, 1.0]])
    return A, B

print(f"{'phi_pi':>7}  {'|omega|':>26}  {'unstable':>8}  verdict")
for phi in (0.0, 0.8, 1.0, 1.5, 2.5):
    Pk, _, om, nu = klein(*nk_matrices(phi), n_x=1)
    verdict = "unique" if nu == 2 else ("indeterminate" if nu < 2 else "no stable solution")
    print(f"{phi:7.1f}  {str(np.round(om, 3)):>26}  {nu:8d}  {verdict}")

**Exercise 4.** The switch happens at $\phi_\pi = 1$. That is the **Taylor principle**, recovered
from nothing but an eigenvalue count. What happens to `P` for $\phi_\pi < 1$, and why is that the
right behaviour rather than a bug?

## Part 4 — Second order, and the risk correction

First order is certainty equivalent: the shock scale enters no coefficient. To get risk you need
the second-order block — twelve conditions in twelve unknowns, exactly as 5.A5 describes.

We impose them directly: write the policies as second-order polynomials in $(\hat{k}, z)$ and
require $F = F_k = F_z = F_{kk} = F_{kz} = F_{zz} = 0$ at the steady state.

In [ ]:
def gauss_hermite(n):
    x, w = np.polynomial.hermite_e.hermegauss(n)
    return x, w/w.sum()


def make_conditions(sigma, order, alpha=ALPHA, beta=BETA, delta=DELTA, rho=RHO, n_gh=7):
    kss, yss, css = steady_state(alpha, beta, delta)
    xe, we = gauss_hermite(n_gh)

    def unpack(th):
        if order == 1:
            return (np.array([th[0], th[1], th[2], 0, 0, 0]),
                    np.array([th[3], th[4], th[5], 0, 0, 0]))
        return np.asarray(th[:6]), np.asarray(th[6:])

    def poly(p, kh, z):
        return p[0] + p[1]*kh + p[2]*z + 0.5*(p[3]*kh**2 + 2*p[4]*kh*z + p[5]*z**2)

    def Ffun(th, kh, z):
        a, b = unpack(th)
        c = max(css*np.exp(np.clip(poly(a, kh, z), -50, 50)), 1e-12)
        khp = poly(b, kh, z)
        kp = max(kss*np.exp(np.clip(khp, -50, 50)), 1e-12)
        k = kss*np.exp(kh)
        f2 = c + kp - np.exp(z)*k**alpha - (1 - delta)*k
        zp = rho*z + sigma*xe
        cp = np.maximum(css*np.exp(np.clip(poly(a, khp, zp), -50, 50)), 1e-12)
        R = alpha*np.exp(zp)*kp**(alpha - 1) + (1 - delta)
        f1 = 1.0/c - beta*np.sum(we*R/cp)
        return np.array([f1, f2])

    h = 1e-5 if order == 1 else 1e-3      # second differences divide by h^2

    def conditions(th):
        f00 = Ffun(th, 0.0, 0.0)
        out = [f00,
               (Ffun(th, h, 0) - Ffun(th, -h, 0))/(2*h),
               (Ffun(th, 0, h) - Ffun(th, 0, -h))/(2*h)]
        if order >= 2:
            out += [(Ffun(th, h, 0) - 2*f00 + Ffun(th, -h, 0))/h**2,
                    (Ffun(th, h, h) - Ffun(th, h, -h)
                     - Ffun(th, -h, h) + Ffun(th, -h, -h))/(4*h**2),
                    (Ffun(th, 0, h) - 2*f00 + Ffun(th, 0, -h))/h**2]
        return np.concatenate(out)

    return conditions


def solve_pert(sigma, order, guess=None):
    cond = make_conditions(sigma, order)
    n = 6 if order == 1 else 12
    if guess is None:
        guess = np.zeros(n)
        guess[1] = guess[4 if order == 1 else 7] = 0.5
    sol = root(cond, guess, method="hybr", tol=1e-13)
    return sol.x, float(np.max(np.abs(cond(sol.x))))

In [ ]:
# first order at sigma = 0 IS the deterministic expansion -- compare with QZ
th0, r0 = solve_pert(0.0, 1)
print(f"matching vs QZ, max abs diff: "
      f"{max(abs(th0[4]-P[0,0]), abs(th0[5]-P[0,1]), abs(th0[1]-F[0,0]), abs(th0[2]-F[0,1])):.3e}")
print(f"constants (certainty equivalence): a0 = {th0[0]:.3e}, b0 = {th0[3]:.3e}")

g = np.concatenate([th0[:3], np.zeros(3), th0[3:], np.zeros(3)])
th2, r2 = solve_pert(SIG_EPS, 2, guess=g)
print(f"\nsecond order, worst condition residual: {r2:.2e}")
print(f"risk correction: consumption {100*th2[0]:+.5f}% of s.s., capital {100*th2[6]:+.5f}%")
print(f"first-order coefficients moved by: "
      f"{max(abs(th2[1]-th0[1]), abs(th2[2]-th0[2]), abs(th2[7]-th0[4]), abs(th2[8]-th0[5])):.2e}")

**Exercise 5.** Two independent routes to the same four numbers — QZ and root-finding on the
derivative conditions — agreeing to about $10^{-11}$. That is the cross-check this course keeps
asking for.

**Exercise 6.** Re-solve at $\sigma \in \{1, 5, 10, 20\}\times$ the calibrated value and plot the
consumption risk correction against $\sigma^2$. The lecture measures the quadratic law holding to
$2\%$ at $5\times$ and failing by $60\%$ at $20\times$. Why does it fail, and what does that tell
you about extrapolating a perturbation result?

**Exercise 7 (trap).** Change `h` in `make_conditions` from `1e-3` to `1e-4` for the second-order
case and re-run. The residual degrades from $\sim 10^{-9}$ to $\sim 10^{-2}$. Second derivatives
divide by $h^2$, so rounding error is amplified by $10^8$ — 4.A4's U-curve, in a place where it
silently corrupts a coefficient. This is why Dynare differentiates analytically.

## Part 5 — Pruning

In [ ]:
def simulate2(b, sigma, T=20000, seed=0, pruned=False):
    rng = np.random.default_rng(seed)
    eps = rng.standard_normal(T)
    z = 0.0
    if not pruned:
        kh, worst = 0.0, 0.0
        for t in range(T):
            z = RHO*z + sigma*eps[t]
            kh = b[0] + b[1]*kh + b[2]*z + 0.5*(b[3]*kh**2 + 2*b[4]*kh*z + b[5]*z**2)
            if not np.isfinite(kh) or abs(kh) > 1e3:
                return np.inf, t
            worst = max(worst, abs(kh))
        return worst, T
    k1 = k2 = worst = 0.0
    for t in range(T):
        z = RHO*z + sigma*eps[t]
        k1, k2 = (b[1]*k1 + b[2]*z,
                  b[0] + b[1]*k2 + 0.5*(b[3]*k1**2 + 2*b[4]*k1*z + b[5]*z**2))
        if not np.isfinite(k1 + k2) or abs(k1 + k2) > 1e3:
            return np.inf, t
        worst = max(worst, abs(k1 + k2))
    return worst, T


print(f"{'sigma x':>8}  {'unpruned':>22}  {'pruned':>10}")
for mult in (1, 10, 30, 60):
    sg = SIG_EPS*mult
    t1, _ = solve_pert(sg, 1)
    gg = np.concatenate([t1[:3], np.zeros(3), t1[3:], np.zeros(3)])
    t2, res = solve_pert(sg, 2, guess=gg)
    wu, tu = simulate2(t2[6:], sg, pruned=False)
    wp, _ = simulate2(t2[6:], sg, pruned=True)
    u = "exploded at t=%d" % tu if not np.isfinite(wu) else "%.4f" % wu
    print(f"{mult:8d}  {u:>22}  {wp:10.4f}")

**Exercise 8.** At the model's own calibration, pruning changes nothing — four figures agree. You
need thirty times the shock before the unpruned recursion blows up. So why prune anyway?

## Part 6 — Chebyshev: why this basis

In [ ]:
# the measured reason not to use monomials
print("Hilbert matrix (the monomial least-squares normal matrix on [0,1]):")
for n in (3, 5, 7, 9, 11, 13):
    H = np.array([[1.0/(i + j + 1) for j in range(n)] for i in range(n)])
    print(f"   n={n:2d}   cond = {np.linalg.cond(H):.3e}")
print("\nDouble precision carries ~16 digits.  Read the last row.")

In [ ]:
print("Interpolation matrix at n Chebyshev nodes:")
print(f"{'n':>4}  {'monomial':>12}  {'Chebyshev':>10}")
for n in (5, 9, 13, 17, 21):
    xs = np.cos(np.pi*(2*np.arange(1, n + 1) - 1)/(2*n))
    Vm = np.vander(xs, n, increasing=True)
    Vc = npcheb.chebvander(xs, n - 1)
    print(f"{n:4d}  {np.linalg.cond(Vm):12.3e}  {np.linalg.cond(Vc):10.4f}")

In [ ]:
# what smoothness buys, and what a kink costs
smooth = lambda x: np.exp(x)*np.sin(3*x)
kinked = lambda x: np.abs(x)
xt = np.linspace(-1, 1, 20001)

print(f"{'n':>4}  {'e^x sin 3x':>14}  {'|x|':>12}")
for n in (5, 9, 17, 33, 65):
    nodes = np.cos(np.pi*(2*np.arange(1, n + 1) - 1)/(2*n))
    row = []
    for f in (smooth, kinked):
        cf = npcheb.chebfit(nodes, f(nodes), n - 1)
        row.append(np.max(np.abs(npcheb.chebval(xt, cf) - f(xt))))
    print(f"{n:4d}  {row[0]:14.3e}  {row[1]:12.3e}")

**Exercise 9.** The analytic function reaches machine precision by $n=33$; $|x|$ halves its error
when $n$ doubles — $O(1/n)$, and sixty-five terms buy two digits. **The basis is not the problem;
the function is.** Which economic models have policy functions like the second column?

## Part 7 — Collocation on the RBC model

In [ ]:
def rouwenhorst(n, rho, sigma):
    p = (1 + rho)/2
    P = np.array([[p, 1 - p], [1 - p, p]])
    for k in range(3, n + 1):
        Z = np.zeros((k, k))
        Z[:-1, :-1] += p*P; Z[:-1, 1:] += (1 - p)*P
        Z[1:, :-1] += (1 - p)*P; Z[1:, 1:] += p*P
        Z[1:-1, :] /= 2
        P = Z
    sz = sigma/np.sqrt(1 - rho**2)
    return np.linspace(-sz*np.sqrt(n - 1), sz*np.sqrt(n - 1), n), P


class Proj:
    def __init__(self, n_coef=7, n_z=7, width=0.5):
        self.alpha, self.beta, self.delta = ALPHA, BETA, DELTA
        self.logz, self.Pi = rouwenhorst(n_z, RHO, SIG_EPS)
        self.z = np.exp(self.logz)
        self.n, self.n_z = n_coef, n_z
        self.kss, _, self.css = steady_state()
        self.klo, self.khi = (1 - width)*self.kss, (1 + width)*self.kss

    def psi(self, k):
        return 2*(k - self.klo)/(self.khi - self.klo) - 1

    def cpol(self, th, k, j):
        return npcheb.chebval(self.psi(np.atleast_1d(k)), th[j])

    def resid(self, th, k, j):
        k = np.atleast_1d(k)
        c = np.maximum(self.cpol(th, k, j), 1e-10)
        y = self.z[j]*k**self.alpha + (1 - self.delta)*k
        kp = np.clip(y - c, self.klo, self.khi)
        rhs = np.zeros_like(k)
        for l in range(self.n_z):
            cp = np.maximum(self.cpol(th, kp, l), 1e-10)
            R = self.alpha*self.z[l]*kp**(self.alpha - 1) + (1 - self.delta)
            rhs += self.Pi[j, l]*R/cp
        return 1.0 - c*self.beta*rhs

    def guess(self):
        """Warm start from the first-order perturbation solution."""
        _, Fm, _, _ = klein(*growth_matrices(), n_x=2)
        nodes = np.cos(np.pi*(2*np.arange(1, self.n + 2) - 1)/(2*(self.n + 1)))
        kn = self.klo + 0.5*(nodes + 1)*(self.khi - self.klo)
        th = np.zeros((self.n_z, self.n + 1))
        for j in range(self.n_z):
            ch = Fm[0, 0]*np.log(kn/self.kss) + Fm[0, 1]*self.logz[j]
            th[j] = npcheb.chebfit(self.psi(kn), self.css*np.exp(ch), self.n)
        return th

    def nodes(self):
        r = np.cos(np.pi*(2*np.arange(1, self.n + 2) - 1)/(2*(self.n + 1)))
        return self.klo + 0.5*(r + 1)*(self.khi - self.klo)

In [ ]:
def solve_collocation(P):
    kn = P.nodes()

    def eqs(flat):
        th = flat.reshape(P.n_z, P.n + 1)
        return np.concatenate([P.resid(th, kn, j) for j in range(P.n_z)])

    t0 = time.perf_counter()
    sol = root(eqs, P.guess().ravel(), method="hybr", tol=1e-12)
    return sol.x.reshape(P.n_z, P.n + 1), time.perf_counter() - t0, sol


def euler_err(cfun, P, n_test=2000, seed=0, band=None):
    """Graded on RANDOM points, never on the collocation nodes."""
    rng = np.random.default_rng(seed)
    lo, hi = (P.klo, P.khi) if band is None else band
    kt = rng.uniform(lo, hi, n_test)
    jt = rng.integers(0, P.n_z, n_test)
    E = np.empty(n_test)
    for i in range(n_test):
        j = int(jt[i]); c = float(cfun(np.array([kt[i]]), j)[0])
        y = P.z[j]*kt[i]**P.alpha + (1 - P.delta)*kt[i]
        kp = min(max(y - c, P.klo), P.khi)
        rhs = 0.0
        for l in range(P.n_z):
            cp = float(cfun(np.array([kp]), l)[0])
            R = P.alpha*P.z[l]*kp**(P.alpha - 1) + (1 - P.delta)
            rhs += P.Pi[j, l]*R/max(cp, 1e-12)
        E[i] = abs(1 - c*P.beta*rhs)
    return E[np.isfinite(E)]


Pj = Proj(n_coef=7)
th, secs, sol = solve_collocation(Pj)
E = euler_err(lambda k, j: Pj.cpol(th, k, j), Pj)
print(f"{(Pj.n+1)*Pj.n_z} unknowns, {secs:.3f} s, "
      f"worst residual at the nodes {np.max(np.abs(sol.fun)):.2e}")
print(f"graded off the nodes: max |E| = 10^{np.log10(E.max()):.2f}")

In [ ]:
# plot the SAVING, not the policy -- a level plot hides everything
kg = np.linspace(Pj.klo, Pj.khi, 300)
jm = Pj.n_z//2
c_mid = Pj.cpol(th, kg, jm)
g_mid = Pj.z[jm]*kg**Pj.alpha + (1 - Pj.delta)*kg - c_mid

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(kg, g_mid - kg, lw=2)
ax.axhline(0, color="0.6", lw=0.8)
ax.axvline(Pj.kss, color="0.6", lw=0.8, ls=":")
ax.set_xlabel("capital $k$"); ax.set_ylabel("saving $g(k)-k$")
ax.set_title("crosses zero at $k^*$, positive below, negative above")
plt.tight_layout(); plt.show()

print("saving at k*:", np.interp(Pj.kss, kg, g_mid - kg))

**Exercise 10.** Sweep the degree over $3, 5, 7, 9, 11$ and tabulate the error. The lecture measures
about **1.2 decades for every two degrees**. Do you reproduce the rate, and does it keep going?

**Exercise 11.** Grade the solution **at the collocation nodes** instead of at random points. You
will get about $10^{-15}$. Explain in one sentence why that number is meaningless.

## Part 8 — One model, four methods

In [ ]:
def c_pert(order):
    if order == 1:
        _, Fm, _, _ = klein(*growth_matrices(), n_x=2)
        return lambda k, j: Pj.css*np.exp(Fm[0, 0]*np.log(np.atleast_1d(k)/Pj.kss)
                                          + Fm[0, 1]*Pj.logz[j])
    a = th2[:6]
    def f(k, j):
        kh = np.log(np.atleast_1d(k)/Pj.kss); z = Pj.logz[j]
        return Pj.css*np.exp(a[0] + a[1]*kh + a[2]*z
                             + 0.5*(a[3]*kh**2 + 2*a[4]*kh*z + a[5]*z**2))
    return f


near = (0.95*Pj.kss, 1.05*Pj.kss)
rows = [("perturbation, 1st order", c_pert(1), 4),
        ("perturbation, 2nd order", c_pert(2), 12),
        ("Chebyshev collocation", lambda k, j: Pj.cpol(th, k, j), (Pj.n+1)*Pj.n_z)]

print(f"{'method':<26} {'numbers':>8}  {'near k*':>10}  {'wide':>10}")
for name, f, store in rows:
    ew = euler_err(f, Pj); en = euler_err(f, Pj, band=near)
    print(f"{name:<26} {store:8d}  10^{np.log10(en.max()):<7.2f}  10^{np.log10(ew.max()):<7.2f}")
print("\nThe lecture's fourth row, the 4.B5 grid solver: 2,100 numbers, "
      "10^-2.82 near k*, 10^-2.51 wide.")

**Exercise 12.** The grid solver stores $37\times$ more numbers than collocation and is four decades
less accurate. Write down the three properties of *this model* that make that true, and name one
model from Session 2 for which each of them fails.

## What to take away

1. **Perturbation changes the representation**, from $N$ values on a grid to a few derivatives at
   one point. The steady state is all it needs, which is why it generalizes.
2. **Blanchard--Kahn is a count**: unstable roots against jump variables. When it fails, the QZ
   routine cannot form a solution — that is the algorithm telling you the truth.
3. **First order is certainty equivalent.** Risk lives in the second-order constant, and it
   scales like $\sigma^2$ only while $\sigma$ is small.
4. **Prune second-order simulations**, even though at realistic calibrations it changes nothing.
   The failure it prevents is silent until it is catastrophic.
5. **Never use monomials.** The Hilbert matrix reaches condition $3\times10^{18}$ at degree 13 and
   the solver returns numbers anyway.
6. **Projection redefines "solved" as "the residual is zero"** — in a chosen sense. The weight
   function matters less than one extra basis coefficient.
7. **Warm-start projection from perturbation.** The two methods cooperate; that is the practical
   reason to know both.
8. **Grade on points you did not solve at.** At the collocation nodes the residual is zero by
   construction, and reporting it is self-deception.
9. **Plot the deviation, not the level.** Again.